# v2 Weights and Hyperparams — Evaluation Notebook

**Purpose.** Narrative wrapper around the Phase 1–4 evaluation pipeline. Reads JSON files from `data/eval/` and produces the figures + tables for Chapter 5 §5.4 / §5.5 / §5.6 of the thesis.

**Run order.** Kernel → Restart & Run All. ~30 sec wallclock on cached data.

**Style.** Mirrors `previous-works/F221611.ipynb`: paired bar charts with value annotations, text-box overlays for headline numbers, `ConfusionMatrixDisplay` with RdGy cmap, `grid(True, alpha=0.3)` everywhere.

**Outputs.**
- `data/eval/figures/*.png` — every plot saved at 200 DPI
- `data/eval/results.md` — markdown tables for the writing chat to lift into the thesis prose

**Phase 4 findings recap (for context as you read the plots):**
1. **Struggle v2** — AUC 0.836 [0.762, 0.911]. Positive: training helps. `n_hat`, `t_hat`, `rep_norm` flipped sign vs v1.
2. **Difficulty v2** — AUC 0.345 (< random). Negative: hand-set v1 near-optimal; LR fits noise at N=72.
3. **Improved-blend v2** — AUC 0.637 (< struggle-v2 alone). Negative: `w_M` and `w_D` flipped negative; BKT/IRT add noise not signal.

## Setup

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EVAL_DIR = REPO_ROOT / 'data' / 'eval'
FIG_DIR = EVAL_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / 'code2'))
from backend import config as _bk_config

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'EVAL_DIR: {EVAL_DIR}')
print(f'FIG_DIR:  {FIG_DIR}')

In [ ]:
def _load(name: str):
    path = EVAL_DIR / name
    if not path.exists():
        print(f'  MISSING: {name}')
        return None
    with path.open(encoding='utf-8') as f:
        return json.load(f)

snapshots_blob   = _load('snapshots.json')
llm_struggle     = _load('llm_struggle_labels.json')
llm_difficulty   = _load('llm_difficulty_labels.json')
self_labels      = _load('self_labels.json')
kappa_report     = _load('kappa_report.json')
opt_struggle     = _load('optimised_struggle_weights_v2.json')
opt_difficulty   = _load('optimised_difficulty_weights_v2.json')
opt_improved     = _load('optimised_improved_weights_v2.json')

snapshots               = snapshots_blob.get('struggle_snapshots', [])
diff_questions          = snapshots_blob.get('difficulty_questions', [])
llm_struggle_labels     = llm_struggle.get('labels', {})
llm_difficulty_labels   = llm_difficulty.get('labels', {})
self_labels_dict        = self_labels.get('labels', {}) if self_labels else {}

print(f'snapshots:             {len(snapshots)}')
print(f'difficulty questions:  {len(diff_questions)}')
print(f'LLM struggle labels:   {len(llm_struggle_labels)}')
print(f'LLM difficulty labels: {len(llm_difficulty_labels)}')
print(f'self labels:           {len(self_labels_dict)}')

In [ ]:
STRUGGLE_BANDS    = ['On Track', 'Minor Issues', 'Struggling', 'Needs Help']
DIFFICULTY_BANDS  = ['Easy', 'Medium', 'Hard', 'Very Hard']
STRUGGLE_SIGNALS  = ['n_hat', 't_hat', 'i_norm', 'r_norm', 'A_norm', 'd_hat', 'rep_norm']
DIFFICULTY_SIGNALS= ['c_norm', 't_tilde', 'a_tilde', 'f_norm', 'p_norm']
IMPROVED_SIGNALS  = ['behavioural_composite', 'mastery_gap', 'difficulty_adjusted_score']

V1_STRUGGLE_WEIGHTS = {
    'n_hat':    _bk_config.STRUGGLE_WEIGHT_N,
    't_hat':    _bk_config.STRUGGLE_WEIGHT_T,
    'i_norm':   _bk_config.STRUGGLE_WEIGHT_I,
    'r_norm':   _bk_config.STRUGGLE_WEIGHT_R,
    'A_norm':   _bk_config.STRUGGLE_WEIGHT_A,
    'd_hat':    _bk_config.STRUGGLE_WEIGHT_D,
    'rep_norm': _bk_config.STRUGGLE_WEIGHT_REP,
}
V1_DIFFICULTY_WEIGHTS = {
    'c_norm':  _bk_config.DIFFICULTY_WEIGHT_C,
    't_tilde': _bk_config.DIFFICULTY_WEIGHT_T,
    'a_tilde': _bk_config.DIFFICULTY_WEIGHT_A,
    'f_norm':  _bk_config.DIFFICULTY_WEIGHT_F,
    'p_norm':  _bk_config.DIFFICULTY_WEIGHT_P,
}
V1_IMPROVED_WEIGHTS = {
    'w_B': _bk_config.IMPROVED_STRUGGLE_WEIGHT_BEHAVIORAL,
    'w_M': _bk_config.IMPROVED_STRUGGLE_WEIGHT_MASTERY_GAP,
    'w_D': _bk_config.IMPROVED_STRUGGLE_WEIGHT_DIFFICULTY_ADJ,
}

COLOR_V1   = '#4169E1'   # royalblue
COLOR_V2   = '#F08080'   # lightcoral
COLOR_OK   = '#10a15d'
COLOR_WARN = '#ff2d55'
COLOR_BAND = '#9CC2E5'

def _save(name: str) -> None:
    plt.tight_layout()
    path = FIG_DIR / f'{name}.png'
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print(f'  saved {path.relative_to(REPO_ROOT)}')

print('V1 struggle weights sum:', sum(V1_STRUGGLE_WEIGHTS.values()))
print('V1 difficulty weights sum:', sum(V1_DIFFICULTY_WEIGHTS.values()))
print('V1 improved blend sum:', sum(V1_IMPROVED_WEIGHTS.values()))

## §5.4.1 — Cohort characteristics

COA122 second-semester deployment. 21 healthy sessions (excluding 2 with <10 students), 1306 student-time snapshots, 72 unique questions. Both v1 thresholds and the GPT-4o-mini rater find the cohort skews high on the struggle and difficulty scales — a publishable finding in itself.

In [ ]:
v1_struggle_bands = Counter(s['v1_struggle_level'] for s in snapshots)
v1_difficulty_bands = Counter(q['v1_difficulty_level'] for q in diff_questions)
llm_struggle_bands = Counter(l['band'] for l in llm_struggle_labels.values())
llm_difficulty_bands = Counter(l['band'] for l in llm_difficulty_labels.values())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (title, counts, bands, total) in zip(axes, [
    ('v1 struggle (snapshots)',    v1_struggle_bands,    STRUGGLE_BANDS,   len(snapshots)),
    ('LLM struggle (snapshots)',   llm_struggle_bands,   STRUGGLE_BANDS,   len(llm_struggle_labels)),
    ('v1 difficulty (questions)',  v1_difficulty_bands,  DIFFICULTY_BANDS, len(diff_questions)),
    ('LLM difficulty (questions)', llm_difficulty_bands, DIFFICULTY_BANDS, len(llm_difficulty_labels)),
]):
    values = [counts.get(b, 0) for b in bands]
    bars = ax.bar(bands, values, color=[COLOR_OK, '#ffcc00', '#ff6600', COLOR_WARN])
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(values)*0.01,
                f'{v}\n({v/total:.0%})' if total else '0', ha='center', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='x', rotation=20)
    ax.set_ylim(0, max(values) * 1.18 if max(values) else 1)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(f'Cohort band distributions — v1 thresholds vs LLM rater', fontsize=13)
_save('cohort_distributions')
plt.show()

## §5.4.2 — Inter-rater agreement (Cohen's κ)

Human (author) self-labelled 50 random snapshots; compared to GPT-4o-mini ratings on the same 50. κ is poor by Landis-Koch but the within-1-band agreement is 70% — raters agree on direction, disagree on the precise band boundary.

In [ ]:
if kappa_report:
    k_intervene = kappa_report['kappa_intervene_unweighted']
    k_band_lin  = kappa_report['kappa_band_linear_weighted']
    k_band_quad = kappa_report['kappa_band_quadratic_weighted']
    n_shared    = kappa_report['n_shared']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    labels = ['intervene\n(unweighted)', 'band\n(linear weights)', 'band\n(quadratic weights)']
    vals   = [k_intervene, k_band_lin, k_band_quad]
    bars   = ax1.bar(labels, vals, color=[COLOR_V1, COLOR_V1, COLOR_V1])
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
    for y, label in [(0.20, 'slight'), (0.40, 'fair'), (0.60, 'moderate'), (0.80, 'substantial')]:
        ax1.axhline(y, color='grey', linestyle='--', alpha=0.5, linewidth=0.7)
        ax1.text(2.55, y+0.005, label, fontsize=8, color='grey', alpha=0.7)
    ax1.set_ylim(-0.05, 1.0)
    ax1.set_ylabel("Cohen's κ")
    ax1.set_title(f"Inter-rater agreement (n={n_shared} shared snapshots)")
    ax1.grid(True, alpha=0.3, axis='y')

    ax2.axis('off')
    summary = [
        ['n shared',            f"{n_shared}"],
        ['intervene κ',         f"{k_intervene:.3f}"],
        ['band linear-w κ',     f"{k_band_lin:.3f}"],
        ['band quadratic-w κ',  f"{k_band_quad:.3f}"],
        ['intervene exact agreement', f"{kappa_report['exact_agreement']['intervene']['rate']:.1%}"],
        ['band exact agreement', f"{kappa_report['exact_agreement']['band']['rate']:.1%}"],
        ['band within-1 agreement', f"{kappa_report['within_1_band_agreement']['rate']:.1%}"],
        ['Human intervene rate', f"{kappa_report['distributions']['human_intervene_rate']:.1%}"],
        ['LLM intervene rate',   f"{kappa_report['distributions']['llm_intervene_rate']:.1%}"],
    ]
    tbl = ax2.table(cellText=summary, colLabels=['metric', 'value'], cellLoc='left', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    ax2.set_title('Agreement breakdown', fontsize=11)

    _save('kappa')
    plt.show()
else:
    print('kappa_report.json missing — skip')

## §5.4.3 — v1 vs v2 weights: STRUGGLE composite (HEADLINE)

Paired bars: v1 hand-set weights vs v2 LR-trained weights. Error bars on v2 from per-fold std (5 folds, session-grouped GroupKFold). **Three signals flipped sign**: `n_hat`, `t_hat`, `rep_norm` — the LR sees engagement-time and submission-volume as proxies for *recovery* rather than *persistence-while-stuck*.

In [ ]:
v1_vals = np.array([V1_STRUGGLE_WEIGHTS[s] for s in STRUGGLE_SIGNALS])
v2_vals = np.array([opt_struggle['weights'][s] for s in STRUGGLE_SIGNALS])
v2_std  = np.array([opt_struggle['weights_per_fold_std'][s] for s in STRUGGLE_SIGNALS])

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(STRUGGLE_SIGNALS))
w = 0.4
bars_v1 = ax.bar(x - w/2, v1_vals, w, label='v1 hand-set', color=COLOR_V1)
bars_v2 = ax.bar(x + w/2, v2_vals, w, yerr=v2_std, capsize=4, label='v2 LR-trained', color=COLOR_V2)
ax.axhline(0, color='black', linewidth=0.6)
for bar, val in zip(bars_v1, v1_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + 0.012 if val >= 0 else val - 0.025,
            f'{val:+.2f}', ha='center', fontsize=9, color=COLOR_V1)
for bar, val in zip(bars_v2, v2_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + 0.012 if val >= 0 else val - 0.025,
            f'{val:+.2f}', ha='center', fontsize=9, color=COLOR_V2)
ax.set_xticks(x)
ax.set_xticklabels(STRUGGLE_SIGNALS)
ax.set_ylabel('Signed weight (L1-normalised for v2)')
ax.set_title(f"v1 hand-set vs v2 LR-trained weights — struggle composite\nv2 AUC = {opt_struggle['auc_mean']:.3f} [{opt_struggle['auc_ci95'][0]:.3f}, {opt_struggle['auc_ci95'][1]:.3f}]")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

flipped = [s for s, v1, v2 in zip(STRUGGLE_SIGNALS, v1_vals, v2_vals) if np.sign(v1) != np.sign(v2)]
flip_text = 'Sign-flipped signals: ' + ', '.join(flipped) if flipped else 'No sign flips'
ax.text(0.02, 0.02, flip_text, transform=ax.transAxes, va='bottom', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='grey'))

_save('weights_struggle_v1_vs_v2')
plt.show()

## §5.4.4 — Per-fold AUC stability (struggle)

Five-fold session-grouped CV. Stability check: per-fold AUC std should be small relative to mean — confirms the v2 weights generalise across held-out sessions rather than overfitting to one cohort.

In [ ]:
fold_aucs = [f['auc'] for f in opt_struggle['per_fold']]
mean_auc  = opt_struggle['auc_mean']
ci_lo, ci_hi = opt_struggle['auc_ci95']

fig, ax = plt.subplots(figsize=(12, 6))
folds = [f"fold {i}\n(C={opt_struggle['per_fold'][i]['best_C']:.3g})" for i in range(len(fold_aucs))]
bars = ax.bar(folds, fold_aucs, color=COLOR_V2)
for bar, val in zip(bars, fold_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}', ha='center', fontsize=10)
ax.axhline(mean_auc, color='black', linestyle='-', linewidth=1, label=f'mean = {mean_auc:.3f}')
ax.axhspan(ci_lo, ci_hi, alpha=0.15, color=COLOR_BAND, label=f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]')
ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, label='chance')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Held-out AUC')
ax.set_title('Per-fold AUC — v2 struggle weights (5 folds, session-grouped CV)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

summary_text = (
    f"Mean AUC: {mean_auc:.3f}\n"
    f"95% CI:   [{ci_lo:.3f}, {ci_hi:.3f}]\n"
    f"Std:      {opt_struggle['auc_std']:.3f}\n"
    f"N folds:  {opt_struggle['n_folds']}\n"
    f"N samples:{opt_struggle['n_samples']}"
)
ax.text(0.02, 0.02, summary_text, transform=ax.transAxes, va='bottom', fontsize=10,
        family='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='grey'))

_save('per_fold_auc_struggle')
plt.show()

## Re-derivation infrastructure (for ROC / calibration / confusion matrices)

The optimised JSONs store per-fold weights but not per-snapshot predictions. The next few plots need predictions, so we re-fit LR per fold using the same `best_C` and pool the held-out predictions. Predictions get cached to `data/eval/pooled_predictions_v2.json` so re-running cells doesn't re-fit.

In [ ]:
def rederive_struggle_v2() -> dict:
    matched = [s for s in snapshots if s['snapshot_id'] in llm_struggle_labels]
    X = np.array([[s['v1_features'][k] for k in STRUGGLE_SIGNALS] for s in matched])
    y = np.array([int(llm_struggle_labels[s['snapshot_id']]['intervene']) for s in matched])
    groups = np.array([s['session_id'] for s in matched])
    pooled = np.zeros(len(matched))
    gkf = GroupKFold(n_splits=5)
    for fold_idx, (tr, te) in enumerate(gkf.split(X, y, groups)):
        best_C = opt_struggle['per_fold'][fold_idx]['best_C']
        sc = StandardScaler().fit(X[tr])
        lr = LogisticRegression(penalty='l2', C=best_C, max_iter=2000,
                                solver='lbfgs', random_state=42)
        lr.fit(sc.transform(X[tr]), y[tr])
        pooled[te] = lr.predict_proba(sc.transform(X[te]))[:, 1]
    return {'matched': matched, 'X': X, 'y': y, 'groups': groups, 'pred': pooled}

def v1_struggle_pooled() -> dict:
    # v1 baseline: read the pre-computed v1_struggle_score from snapshots (already includes shrinkage)
    matched = [s for s in snapshots if s['snapshot_id'] in llm_struggle_labels]
    pred = np.array([s['v1_struggle_score'] for s in matched])
    y    = np.array([int(llm_struggle_labels[s['snapshot_id']]['intervene']) for s in matched])
    return {'matched': matched, 'pred': pred, 'y': y}

v2_struggle = rederive_struggle_v2()
v1_struggle = v1_struggle_pooled()

v1_auc = roc_auc_score(v1_struggle['y'], v1_struggle['pred'])
v2_auc = roc_auc_score(v2_struggle['y'], v2_struggle['pred'])
print(f'v1 baseline AUC (pooled, intervene): {v1_auc:.3f}')
print(f'v2 trained AUC (pooled, intervene):  {v2_auc:.3f}   (matches stored {opt_struggle["auc_mean"]:.3f}? {abs(v2_auc - opt_struggle["auc_mean"]) < 0.02})')

# Cache for future cells
pooled_cache = {
    'v1_struggle': {'pred': v1_struggle['pred'].tolist(), 'y': v1_struggle['y'].tolist()},
    'v2_struggle': {'pred': v2_struggle['pred'].tolist(), 'y': v2_struggle['y'].tolist()},
    'snapshot_ids': [s['snapshot_id'] for s in v2_struggle['matched']],
}
(EVAL_DIR / 'pooled_predictions_v2.json').write_text(json.dumps(pooled_cache, indent=2), encoding='utf-8')
print(f'  cached pooled predictions to data/eval/pooled_predictions_v2.json')

## §5.4.5 — ROC curves

v1 baseline (shrunk struggle score) and v2 trained, both scored against the LLM intervene label on the same 1306 snapshots. AUC values in the legend.

In [ ]:
fpr1, tpr1, _ = roc_curve(v1_struggle['y'], v1_struggle['pred'])
fpr2, tpr2, _ = roc_curve(v2_struggle['y'], v2_struggle['pred'])

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr1, tpr1, '-',  color=COLOR_V1, linewidth=2, label=f'v1 hand-set  (AUC = {v1_auc:.3f})')
ax.plot(fpr2, tpr2, '--', color=COLOR_V2, linewidth=2, label=f'v2 trained  (AUC = {v2_auc:.3f})')
ax.plot([0, 1], [0, 1], ':', color='grey', linewidth=1, label='chance (AUC 0.5)')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC — v1 vs v2 struggle composite (LLM intervene target)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
_save('roc_struggle')
plt.show()

## §5.4.6 — Calibration (reliability diagram)

10-bin reliability diagram for the v2 struggle scores. Shows whether predicted probability can be read as a probability of intervention need, or only as a relative ranking score.

In [ ]:
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_idx = np.digitize(v2_struggle['pred'], bins) - 1
bin_idx = np.clip(bin_idx, 0, n_bins - 1)

bin_pred_means = []
bin_obs_rates  = []
bin_counts     = []
for i in range(n_bins):
    mask = bin_idx == i
    if mask.sum() == 0:
        bin_pred_means.append(np.nan)
        bin_obs_rates.append(np.nan)
        bin_counts.append(0)
    else:
        bin_pred_means.append(v2_struggle['pred'][mask].mean())
        bin_obs_rates.append(v2_struggle['y'][mask].mean())
        bin_counts.append(int(mask.sum()))

brier = brier_score_loss(v2_struggle['y'], v2_struggle['pred'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'height_ratios': None, 'width_ratios': [3, 2]})
ax1.plot([0, 1], [0, 1], ':', color='grey', label='perfect calibration')
ax1.plot(bin_pred_means, bin_obs_rates, 'o-', color=COLOR_V2, linewidth=2, markersize=8,
         label=f'v2 struggle (Brier = {brier:.3f})')
ax1.set_xlabel('Mean predicted probability per bin')
ax1.set_ylabel('Observed positive rate per bin')
ax1.set_title('Reliability diagram — v2 struggle (10 bins)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2.bar(bin_centers, bin_counts, width=0.08, color=COLOR_V2, edgecolor='black')
ax2.set_xlabel('Predicted probability bin')
ax2.set_ylabel('Snapshots in bin')
ax2.set_title('Prediction histogram')
ax2.grid(True, alpha=0.3, axis='y')

_save('calibration_struggle')
plt.show()

## §5.4.7 — Per-fold weight stability (heatmap)

7 signals × 5 folds. Cell colour = signed weight, annotated. If folds agree on sign and magnitude, the v2 weight vector is reproducible; large disagreements would suggest the LR is fitting noise.

In [ ]:
fold_weights = np.array([
    [f['weights_convex'][s] for s in STRUGGLE_SIGNALS]
    for f in opt_struggle['per_fold']
])

fig, ax = plt.subplots(figsize=(10, 6))
vmax = np.abs(fold_weights).max()
im = ax.imshow(fold_weights.T, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(opt_struggle['per_fold'])))
ax.set_xticklabels([f"fold {i}" for i in range(len(opt_struggle['per_fold']))])
ax.set_yticks(range(len(STRUGGLE_SIGNALS)))
ax.set_yticklabels(STRUGGLE_SIGNALS)
for i in range(fold_weights.shape[1]):  # signal
    for j in range(fold_weights.shape[0]):  # fold
        val = fold_weights[j, i]
        color = 'white' if abs(val) > vmax * 0.5 else 'black'
        ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=10, color=color)
ax.set_title('Per-fold weight stability — v2 struggle (convex normalised)')
fig.colorbar(im, ax=ax, label='Signed weight (L1-norm)')
_save('weight_heatmap_struggle')
plt.show()

## §5.4.8 — Confusion matrix: v1 / v2 predicted bands vs LLM bands

Map pooled predictions to the 4 struggle bands (using `STRUGGLE_THRESHOLDS` from config) and compare to the LLM-rated band on the same snapshot. Quantifies where v1 / v2 over- or under-classify struggle severity vs. the second-opinion rater.

In [ ]:
STRUGGLE_THRESHOLDS = _bk_config.STRUGGLE_THRESHOLDS

def score_to_band(score: float) -> str:
    for low, high, label, _ in STRUGGLE_THRESHOLDS:
        if low <= score < high:
            return label
    return STRUGGLE_THRESHOLDS[-1][2]

matched = v2_struggle['matched']
llm_bands_arr = np.array([llm_struggle_labels[s['snapshot_id']]['band'] for s in matched])
v1_bands_arr  = np.array([score_to_band(p) for p in v1_struggle['pred']])
v2_bands_arr  = np.array([score_to_band(p) for p in v2_struggle['pred']])

cm_v1 = confusion_matrix(llm_bands_arr, v1_bands_arr, labels=STRUGGLE_BANDS)
cm_v2 = confusion_matrix(llm_bands_arr, v2_bands_arr, labels=STRUGGLE_BANDS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for ax, cm, title in [(ax1, cm_v1, 'v1 hand-set'), (ax2, cm_v2, 'v2 trained')]:
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=STRUGGLE_BANDS)
    disp.plot(ax=ax, cmap='RdGy', values_format='d', xticks_rotation=20, colorbar=False)
    ax.set_title(f'{title}: predicted band vs LLM band')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('LLM (truth)')
fig.suptitle('Band-classification confusion matrices')
_save('confusion_bands')
plt.show()

agree_v1 = int(np.diag(cm_v1).sum())
agree_v2 = int(np.diag(cm_v2).sum())
total = int(cm_v1.sum())
print(f'v1 exact-band agreement with LLM: {agree_v1}/{total} ({agree_v1/total:.1%})')
print(f'v2 exact-band agreement with LLM: {agree_v2}/{total} ({agree_v2/total:.1%})')

## §5.4.9 — Negative findings: difficulty + improved-blend

Two paired bar charts mirroring §5.4.3 but for the composites where v2 *underperforms* v1.

- **Difficulty v2** (AUC 0.345) cannot discriminate Very-Hard from Hard at N=72 — LR fits noise. Weights show counter-intuitive negative signs on `t_tilde` and `a_tilde`.
- **Improved-blend v2** (AUC 0.637) assigns negative weight to `w_M` (BKT mastery-gap) and `w_D` (IRT-adjusted exposure) — the LR concludes these components add noise rather than signal, at least for predicting acute intervention need.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Difficulty ---
ax = axes[0]
v1d = np.array([V1_DIFFICULTY_WEIGHTS[s] for s in DIFFICULTY_SIGNALS])
v2d = np.array([opt_difficulty['weights'][s] for s in DIFFICULTY_SIGNALS])
v2d_std = np.array([opt_difficulty['weights_per_fold_std'][s] for s in DIFFICULTY_SIGNALS])
x = np.arange(len(DIFFICULTY_SIGNALS))
w = 0.4
ax.bar(x - w/2, v1d, w, label='v1 hand-set', color=COLOR_V1)
ax.bar(x + w/2, v2d, w, yerr=v2d_std, capsize=4, label='v2 LR-trained', color=COLOR_V2)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels(DIFFICULTY_SIGNALS)
ax.set_ylabel('Signed weight')
ax.set_title(f"DIFFICULTY: v1 vs v2\nv2 AUC = {opt_difficulty['auc_mean']:.3f}  —  NEGATIVE FINDING (v1 wins)")
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.text(0.02, 0.02, 'v2 AUC < 0.5: v1 hand-set already near-optimal at N=72.\nLR fits noise on COA122 uniformly-hard cohort.',
        transform=ax.transAxes, va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=COLOR_WARN))

# --- Improved blend ---
ax = axes[1]
IMP_LABELS = ['w_B\n(behavioural)', 'w_M\n(mastery gap)', 'w_D\n(IRT-adjusted)']
v1i = np.array([V1_IMPROVED_WEIGHTS[k] for k in ['w_B', 'w_M', 'w_D']])
v2i = np.array([opt_improved['weights'][k] for k in ['w_B', 'w_M', 'w_D']])
v2i_std = np.array([opt_improved['weights_per_fold_std'][k] for k in ['w_B', 'w_M', 'w_D']])
x = np.arange(3)
ax.bar(x - w/2, v1i, w, label='v1 hand-set', color=COLOR_V1)
ax.bar(x + w/2, v2i, w, yerr=v2i_std, capsize=4, label='v2 LR-trained', color=COLOR_V2)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels(IMP_LABELS)
ax.set_ylabel('Signed blend weight')
ax.set_title(f"IMPROVED BLEND: v1 vs v2\nv2 AUC = {opt_improved['auc_mean']:.3f}  —  NEGATIVE FINDING (v1 blend wins)")
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.text(0.02, 0.02, 'w_M and w_D flipped NEGATIVE.\nLR rejects BKT mastery + IRT exposure as predictors of intervene.',
        transform=ax.transAxes, va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=COLOR_WARN))

fig.suptitle('Negative findings — v2 weights worse than v1 (kept as research artefacts)', fontsize=12)
_save('negative_findings')
plt.show()

## §5.6.1 — Model disagreement matrix (v1 ↔ v2 bands on shared snapshots)

Predicted struggle bands under v1 vs under v2 for the same 1306 snapshots. The off-diagonals show where v2 reclassifies students; the diagonal shows agreement. Headline number: percentage of snapshots whose band changes under v2.

In [ ]:
cm_v1v2 = confusion_matrix(v1_bands_arr, v2_bands_arr, labels=STRUGGLE_BANDS)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_v1v2, display_labels=STRUGGLE_BANDS)
disp.plot(ax=ax, cmap='RdGy', values_format='d', xticks_rotation=20, colorbar=False)
ax.set_xlabel('v2 predicted band')
ax.set_ylabel('v1 predicted band')
ax.set_title('Model-disagreement matrix — v1 vs v2 (same snapshots)')
_save('model_disagreement')
plt.show()

n_same = int(np.diag(cm_v1v2).sum())
n_total = int(cm_v1v2.sum())
n_diff = n_total - n_same
print(f'Same band under v1 and v2: {n_same}/{n_total} ({n_same/n_total:.1%})')
print(f'Reclassified under v2:     {n_diff}/{n_total} ({n_diff/n_total:.1%})')

upgrades = 0
downgrades = 0
for i, vi_band in enumerate(STRUGGLE_BANDS):
    for j, vj_band in enumerate(STRUGGLE_BANDS):
        if i == j: continue
        # Higher index = more severe band; v2 j>i means v2 sees it as worse
        if j > i:
            upgrades += cm_v1v2[i, j]
        else:
            downgrades += cm_v1v2[i, j]
print(f'Upgraded (v2 sees as more severe):   {upgrades} ({upgrades/n_total:.1%})')
print(f'Downgraded (v2 sees as less severe): {downgrades} ({downgrades/n_total:.1%})')

## §5.4.10 — Hyperparam optimisation (Optuna TPE)

Two parallel TPE studies, 50 trials each, session-grouped 5-fold CV against LLM intervene labels. Joint plot of trajectories (trial number vs AUC) and optimisation landscape (parameter value vs AUC).

**Headline**:
- **Shrinkage K**: best=1, AUC=0.805, Δ=+0.007 vs v1 K=5 → robustness finding (hand-set near-optimal)
- **CF τ**: best=0.899, AUC=0.800, Δ=+0.118 vs v1 τ=0.7 → substantial positive (hand-set τ was too permissive)

**Caveat**: best τ landed at the upper boundary of the [0.4, 0.9] search range — finer search may yield further improvement. Reported in §5.5.

In [ ]:
opt_hp = _load('optimised_hyperparams_v2.json')
if opt_hp is None:
    print('optimised_hyperparams_v2.json missing — run scripts/optimise_hyperparams.py first')
else:
    # Extract per-trial data for both studies
    k_trials = opt_hp['studies']['shrinkage_k']['trials']
    t_trials = opt_hp['studies']['cf_threshold']['trials']
    k_x = [t['number'] for t in k_trials]
    k_y = [t['value'] for t in k_trials]
    k_param = [t['params']['shrinkage_k'] for t in k_trials]
    t_x = [t['number'] for t in t_trials]
    t_y = [t['value'] for t in t_trials]
    t_param = [t['params']['cf_threshold'] for t in t_trials]

    best_k       = opt_hp['best_values']['shrinkage_k']
    best_t       = opt_hp['best_values']['cf_threshold']
    best_k_auc   = opt_hp['best_aucs']['shrinkage_k_best']
    best_t_auc   = opt_hp['best_aucs']['cf_threshold_best']
    base_k_auc   = opt_hp['v1_baseline_aucs']['shrinkage_k_at_default']
    base_t_auc   = opt_hp['v1_baseline_aucs']['cf_threshold_at_default']
    default_k    = opt_hp['v1_baseline_aucs']['default_k']
    default_tau  = opt_hp['v1_baseline_aucs']['default_tau']

    # Optuna's "best-so-far" running maximum — clearer than raw trial values
    k_best_so_far = np.maximum.accumulate(k_y)
    t_best_so_far = np.maximum.accumulate(t_y)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # --- (0, 0): K trajectory ---
    ax = axes[0, 0]
    ax.plot(k_x, k_y, 'o', color=COLOR_V2, alpha=0.4, markersize=5, label='trial AUC')
    ax.plot(k_x, k_best_so_far, '-', color=COLOR_V2, linewidth=2, label='best so far')
    ax.axhline(base_k_auc, color=COLOR_V1, linestyle='--', linewidth=1.2,
               label=f'v1 default (K={default_k}) = {base_k_auc:.3f}')
    ax.set_xlabel('Trial number'); ax.set_ylabel('CV AUC')
    ax.set_title(f'Shrinkage K — Optuna trajectory\nbest K={best_k}, AUC={best_k_auc:.3f}')
    ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)

    # --- (0, 1): τ trajectory ---
    ax = axes[0, 1]
    ax.plot(t_x, t_y, 'o', color=COLOR_V2, alpha=0.4, markersize=5, label='trial AUC')
    ax.plot(t_x, t_best_so_far, '-', color=COLOR_V2, linewidth=2, label='best so far')
    ax.axhline(base_t_auc, color=COLOR_V1, linestyle='--', linewidth=1.2,
               label=f'v1 default (τ={default_tau}) = {base_t_auc:.3f}')
    ax.set_xlabel('Trial number'); ax.set_ylabel('CV AUC')
    ax.set_title(f'CF threshold τ — Optuna trajectory\nbest τ={best_t:.3f}, AUC={best_t_auc:.3f}')
    ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)

    # --- (1, 0): K vs AUC landscape ---
    ax = axes[1, 0]
    ax.scatter(k_param, k_y, color=COLOR_V2, alpha=0.6, s=40)
    ax.scatter([best_k], [best_k_auc], color=COLOR_OK, s=180, marker='*',
               edgecolor='black', linewidth=1.0, zorder=5, label=f'best (K={best_k})')
    ax.scatter([default_k], [base_k_auc], color=COLOR_V1, s=120, marker='D',
               edgecolor='black', linewidth=1.0, zorder=5, label=f'v1 default (K={default_k})')
    ax.set_xlabel('Shrinkage K'); ax.set_ylabel('CV AUC')
    ax.set_title('Shrinkage K — optimisation landscape')
    ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)

    # --- (1, 1): τ vs AUC landscape ---
    ax = axes[1, 1]
    ax.scatter(t_param, t_y, color=COLOR_V2, alpha=0.6, s=40)
    ax.scatter([best_t], [best_t_auc], color=COLOR_OK, s=180, marker='*',
               edgecolor='black', linewidth=1.0, zorder=5, label=f'best (τ={best_t:.3f})')
    ax.scatter([default_tau], [base_t_auc], color=COLOR_V1, s=120, marker='D',
               edgecolor='black', linewidth=1.0, zorder=5, label=f'v1 default (τ={default_tau})')
    ax.set_xlabel('CF threshold τ'); ax.set_ylabel('CV AUC')
    ax.set_title('CF threshold τ — optimisation landscape')
    ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)

    summary = (
        f"K:    best={best_k}   AUC={best_k_auc:.3f}   Δ vs v1 = {best_k_auc - base_k_auc:+.3f}\n"
        f"τ:    best={best_t:.3f}  AUC={best_t_auc:.3f}   Δ vs v1 = {best_t_auc - base_t_auc:+.3f}\n"
        f"50 trials per study via Optuna TPE (seed=42)"
    )
    fig.suptitle('Hyperparam optimisation — Optuna TPE results', fontsize=13)
    fig.text(0.5, -0.01, summary, ha='center', fontsize=10, family='monospace',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='grey'))

    _save('hyperparams_optuna')
    plt.show()

## Markdown export — `data/eval/results.md`

Final cell writes a Markdown file with all the headline numbers + tables, ready for the writing chat to lift verbatim into Chapter 5 §5.4 / §5.5 / §5.6 prose.

In [ ]:
lines = []
A = lines.append

A('# Evaluation Results — v2 Weights and Hyperparams')
A('')
A('Auto-generated by `notebooks/eval_main.ipynb`. Lift these tables and numbers directly into Chapter 5 §5.4 / §5.5 / §5.6 prose.')
A('')

A('## Cohort')
A('| Metric | Value |')
A('|---|---|')
A(f'| Snapshots | {len(snapshots):,} |')
A(f'| Unique questions | {len(diff_questions)} |')
A(f'| Healthy sessions | 21 of 23 |')
A(f'| Time range | 2025-10-06 → 2026-05-15 |')
A('')

A("## §5.4.2 Inter-rater agreement (Cohen's κ, n=50 shared snapshots)")
if kappa_report:
    A('| Metric | Value |')
    A('|---|---|')
    A(f"| κ intervene (unweighted) | **{kappa_report['kappa_intervene_unweighted']:.3f}** |")
    A(f"| κ band (linear weights) | **{kappa_report['kappa_band_linear_weighted']:.3f}** |")
    A(f"| κ band (quadratic weights) | **{kappa_report['kappa_band_quadratic_weighted']:.3f}** |")
    A(f"| Intervene exact agreement | {kappa_report['exact_agreement']['intervene']['rate']:.1%} |")
    A(f"| Band exact agreement | {kappa_report['exact_agreement']['band']['rate']:.1%} |")
    A(f"| Band within-1-step agreement | **{kappa_report['within_1_band_agreement']['rate']:.1%}** |")
    A(f"| Human intervene rate | {kappa_report['distributions']['human_intervene_rate']:.1%} |")
    A(f"| LLM intervene rate | {kappa_report['distributions']['llm_intervene_rate']:.1%} |")
    A('')

A('## §5.4.3 Struggle composite: v1 vs v2 weights (HEADLINE POSITIVE)')
A(f"v2 mean AUC = **{opt_struggle['auc_mean']:.3f}** [{opt_struggle['auc_ci95'][0]:.3f}, {opt_struggle['auc_ci95'][1]:.3f}] across 5 session-grouped folds.")
A('')
A('| Signal | v1 (hand-set) | v2 (trained) | v2 per-fold std | Sign |')
A('|---|---|---|---|---|')
for s in STRUGGLE_SIGNALS:
    v1w = V1_STRUGGLE_WEIGHTS[s]
    v2w = opt_struggle['weights'][s]
    std = opt_struggle['weights_per_fold_std'][s]
    flipped = '**FLIPPED**' if np.sign(v1w) != np.sign(v2w) else ''
    A(f'| `{s}` | {v1w:+.3f} | {v2w:+.3f} | ±{std:.3f} | {flipped} |')
A('')

A('## §5.4.4 Per-fold AUC (struggle)')
A('| Fold | C | AUC | n_train | n_test | pos rate |')
A('|---|---|---|---|---|---|')
for i, f in enumerate(opt_struggle['per_fold']):
    A(f"| {i} | {f['best_C']:.3g} | {f['auc']:.3f} | {f['n_train']} | {f['n_test']} | {f['positive_rate_test']:.1%} |")
A('')

A('## §5.4.9 Difficulty composite: v1 vs v2 weights (NEGATIVE finding)')
A(f"v2 pooled AUC = **{opt_difficulty['auc_mean']:.3f}** (LOO on questions, N={opt_difficulty['n_samples']}).")
A('Hand-set v1 is near-optimal for this cohort — LR cannot discriminate Very-Hard from Hard given 5 saturating features.')
A('')
A('| Signal | v1 (hand-set) | v2 (trained) | v2 per-fold std | Sign |')
A('|---|---|---|---|---|')
for s in DIFFICULTY_SIGNALS:
    v1w = V1_DIFFICULTY_WEIGHTS[s]
    v2w = opt_difficulty['weights'][s]
    std = opt_difficulty['weights_per_fold_std'][s]
    flipped = '**FLIPPED**' if np.sign(v1w) != np.sign(v2w) else ''
    A(f'| `{s}` | {v1w:+.3f} | {v2w:+.3f} | ±{std:.3f} | {flipped} |')
A('')

A('## §5.4.9 Improved-struggle blend: v1 vs v2 (NEGATIVE finding)')
A(f"v2 mean AUC = **{opt_improved['auc_mean']:.3f}** [{opt_improved['auc_ci95'][0]:.3f}, {opt_improved['auc_ci95'][1]:.3f}] — worse than baseline struggle v2 alone (0.836).")
A('LR assigned NEGATIVE weights to BKT mastery-gap and IRT-adjusted exposure components.')
A('')
A('| Weight | v1 (hand-set) | v2 (trained) | v2 per-fold std | Sign |')
A('|---|---|---|---|---|')
for k in ['w_B', 'w_M', 'w_D']:
    v1w = V1_IMPROVED_WEIGHTS[k]
    v2w = opt_improved['weights'][k]
    std = opt_improved['weights_per_fold_std'][k]
    flipped = '**FLIPPED**' if np.sign(v1w) != np.sign(v2w) else ''
    A(f'| `{k}` | {v1w:+.3f} | {v2w:+.3f} | ±{std:.3f} | {flipped} |')
A('')

A('## §5.4.10 Hyperparam optimisation (Optuna TPE)')
if opt_hp:
    A(f"50 trials per study, session-grouped 5-fold CV against LLM intervene labels.")
    A('')
    A('| Hyperparam | v1 default | v1 CV AUC | v2 best | v2 CV AUC | Δ AUC | Verdict |')
    A('|---|---|---|---|---|---|---|')
    bk = opt_hp['best_values']['shrinkage_k']
    bt = opt_hp['best_values']['cf_threshold']
    bk_auc = opt_hp['best_aucs']['shrinkage_k_best']
    bt_auc = opt_hp['best_aucs']['cf_threshold_best']
    base_k_auc = opt_hp['v1_baseline_aucs']['shrinkage_k_at_default']
    base_t_auc = opt_hp['v1_baseline_aucs']['cf_threshold_at_default']
    dk = opt_hp['v1_baseline_aucs']['default_k']
    dt = opt_hp['v1_baseline_aucs']['default_tau']
    A(f"| `shrinkage_k` | {dk} | {base_k_auc:.3f} | **{bk}** | {bk_auc:.3f} | **{bk_auc - base_k_auc:+.3f}** | Robustness (hand-set near-optimal) |")
    A(f"| `cf_threshold` | {dt} | {base_t_auc:.3f} | **{bt:.3f}** | {bt_auc:.3f} | **{bt_auc - base_t_auc:+.3f}** | Substantial positive (hand-set was too permissive) |")
    A('')
    A(f"**Caveat for §5.5**: best τ landed at the upper boundary of the [0.4, 0.9] search range — finer search may yield further improvement.")
    A('')
    A('**Deferred** (require ~30 min of BKT refits per trial): BKT priors `(p_init, p_learn, p_guess, p_slip)` + BKT mastery threshold. Pinned to `config.py` defaults in the v2 hyperparams JSON.')
    A('')

A('## §5.6.1 Model disagreement (v1 ↔ v2 struggle bands, n=1306)')
A(f'- Same band under both: **{n_same}/{n_total} ({n_same/n_total:.1%})**')
A(f'- Reclassified under v2: **{n_diff}/{n_total} ({n_diff/n_total:.1%})**')
A(f'- Upgraded (v2 sees as more severe): {upgrades} ({upgrades/n_total:.1%})')
A(f'- Downgraded (v2 sees as less severe): {downgrades} ({downgrades/n_total:.1%})')
A('')

A('---')
A(f'_Generated by `notebooks/eval_main.ipynb` on {pd.Timestamp.now(tz="UTC").isoformat()}_.')

(EVAL_DIR / 'results.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote {(EVAL_DIR / "results.md")}  ({len(lines)} lines)')